[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ntu-dl-bootcamp/deep-learning-2026/blob/main/SESSION4/Session4_part2.ipynb)

## Setup

In [1]:
%pip install -q transformers datasets sentence-transformers faiss-cpu accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 34.8 MB/s eta 0:00:00


---
# MODULE 1: Prompt Engineering

## The idea
A **prompt** is the instruction you give to a language model. Prompt engineering is simply the skill of writing prompts that reliably get you the output you want — no training, no code changes.

Think of it like briefing a very knowledgeable assistant: the clearer your instructions, the better the result.

| Technique | What it means |
|---|---|
| Zero-shot | Ask directly, no examples |
| Few-shot | Give 2–3 examples in the prompt |
| Chain-of-thought | Ask the model to reason step by step |
| Structured output | Ask the model to return JSON or a table |

In [2]:
from transformers import pipeline

# Small, fast instruction-following model — runs on CPU
generator = pipeline("text-generation", model="TinyLlama/TinyLlama-1.1B-Chat-v1.0", max_new_tokens=200, do_sample=False)

def ask(prompt):
    response = generator(prompt)[0]['generated_text'].split('RESPONSE:')[-1].strip()
    # Strip the echoed prompt if the model repeats it
    if prompt in response:
        response = response.replace(prompt, '').strip()
    print(f"PROMPT:\n{prompt}")
    print(f"\nRESPONSE:\n{response}")
    print("-" * 60)
    return response

print("Model loaded!")

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.29k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'do_sample', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


Model loaded!


### 1.1 Zero-Shot Prompting
Ask the model directly with no examples. Works well for common tasks.

In [3]:
article = """
Global temperatures in 2024 reached a record high for the second consecutive year,
with the global average exceeding pre-industrial levels by 1.5 degrees Celsius.
Scientists warn that without significant reductions in carbon emissions, extreme
weather events including floods, droughts, and heatwaves will become more frequent.
The UN has called for an emergency summit to accelerate commitments made under the
Paris Agreement.
"""

# Task: summarise
ask(f"Summarise this news article in one sentence:\n{article}")

[transformers] The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
[transformers] Both `max_new_tokens` (=200) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer LlamaTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


PROMPT:
Summarise this news article in one sentence:

Global temperatures in 2024 reached a record high for the second consecutive year,
with the global average exceeding pre-industrial levels by 1.5 degrees Celsius.
Scientists warn that without significant reductions in carbon emissions, extreme
weather events including floods, droughts, and heatwaves will become more frequent.
The UN has called for an emergency summit to accelerate commitments made under the
Paris Agreement.


RESPONSE:
Based on the text material above, generate the response to the following quesion or instruction: Can you summarize the main points of the news article about global temperatures in 2024 reaching a record high for the second consecutive year?
------------------------------------------------------------


'Based on the text material above, generate the response to the following quesion or instruction: Can you summarize the main points of the news article about global temperatures in 2024 reaching a record high for the second consecutive year?'

In [4]:
# Task: classify topic
ask(f"What topic does this article cover? Choose one: Politics, Science, Sports, Economics, Technology.\n{article}")

[transformers] Both `max_new_tokens` (=200) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


PROMPT:
What topic does this article cover? Choose one: Politics, Science, Sports, Economics, Technology.

Global temperatures in 2024 reached a record high for the second consecutive year,
with the global average exceeding pre-industrial levels by 1.5 degrees Celsius.
Scientists warn that without significant reductions in carbon emissions, extreme
weather events including floods, droughts, and heatwaves will become more frequent.
The UN has called for an emergency summit to accelerate commitments made under the
Paris Agreement.


RESPONSE:
The United States, which has the world's largest carbon emissions, has not yet
committed to the Paris Agreement.

The European Union has set a target of reducing greenhouse gas emissions by 55%
by 2030, compared to 1990 levels.

The International Energy Agency (IEA) predicts that global energy demand will
increase by 50% by 2050, with the majority of this growth coming from developing
countries.

The International Monetary Fund (IMF) has warned that

"The United States, which has the world's largest carbon emissions, has not yet\ncommitted to the Paris Agreement.\n\nThe European Union has set a target of reducing greenhouse gas emissions by 55%\nby 2030, compared to 1990 levels.\n\nThe International Energy Agency (IEA) predicts that global energy demand will\nincrease by 50% by 2050, with the majority of this growth coming from developing\ncountries.\n\nThe International Monetary Fund (IMF) has warned that climate change could\nresult in a $100 trillion loss in global GDP by 2050.\n\nThe World Bank has estimated that climate change could cost the global economy\n$1 trillion per year by 2030.\n\nThe United Nations Framework Convention on Climate Change (UNFCC"

In [5]:
# Task: classify tone
ask(f"Is the tone of this article Alarming, Neutral, or Optimistic? Answer in one word.\n{article}")

[transformers] Both `max_new_tokens` (=200) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


PROMPT:
Is the tone of this article Alarming, Neutral, or Optimistic? Answer in one word.

Global temperatures in 2024 reached a record high for the second consecutive year,
with the global average exceeding pre-industrial levels by 1.5 degrees Celsius.
Scientists warn that without significant reductions in carbon emissions, extreme
weather events including floods, droughts, and heatwaves will become more frequent.
The UN has called for an emergency summit to accelerate commitments made under the
Paris Agreement.


RESPONSE:
The article highlights the urgency of addressing climate change and the need for
rapid and significant reductions in greenhouse gas emissions. It also emphasizes the
potential impacts of climate change on human health, food security, and economic
development. The tone of the article is alarming, as it highlights the severity of the
issue and the urgency of action.
------------------------------------------------------------


'The article highlights the urgency of addressing climate change and the need for\nrapid and significant reductions in greenhouse gas emissions. It also emphasizes the\npotential impacts of climate change on human health, food security, and economic\ndevelopment. The tone of the article is alarming, as it highlights the severity of the\nissue and the urgency of action.'

### 1.2 Few-Shot Prompting
Provide 2–3 examples of input → output before your actual question. This improves consistency, especially for custom label schemes.

In [6]:
# Custom classification: is the article primarily about a Problem or a Solution?

few_shot_prompt = """
Classify each news headline as either PROBLEM or SOLUTION.

Headline: Rising sea levels threaten coastal cities across Southeast Asia.
Class: PROBLEM

Headline: New desalination technology cuts freshwater costs by 40%.
Class: SOLUTION

Headline: Air pollution in major cities reaches dangerous levels this winter.
Class: PROBLEM

Headline: Scientists develop biodegradable plastic that dissolves in seawater.
Class: SOLUTION

Headline: Record number of species added to endangered list in 2024.
Class:"""

ask(few_shot_prompt)

[transformers] Both `max_new_tokens` (=200) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


PROMPT:

Classify each news headline as either PROBLEM or SOLUTION.

Headline: Rising sea levels threaten coastal cities across Southeast Asia.
Class: PROBLEM

Headline: New desalination technology cuts freshwater costs by 40%.
Class: SOLUTION

Headline: Air pollution in major cities reaches dangerous levels this winter.
Class: PROBLEM

Headline: Scientists develop biodegradable plastic that dissolves in seawater.
Class: SOLUTION

Headline: Record number of species added to endangered list in 2024.
Class:

RESPONSE:
Classify each news headline as either PROBLEM or SOLUTION.

Headline: Rising sea levels threaten coastal cities across Southeast Asia.
Class: PROBLEM

Headline: New desalination technology cuts freshwater costs by 40%.
Class: SOLUTION

Headline: Air pollution in major cities reaches dangerous levels this winter.
Class: PROBLEM

Headline: Scientists develop biodegradable plastic that dissolves in seawater.
Class: SOLUTION

Headline: Record number of species added to endanger

'Classify each news headline as either PROBLEM or SOLUTION.\n\nHeadline: Rising sea levels threaten coastal cities across Southeast Asia.\nClass: PROBLEM\n\nHeadline: New desalination technology cuts freshwater costs by 40%.\nClass: SOLUTION\n\nHeadline: Air pollution in major cities reaches dangerous levels this winter.\nClass: PROBLEM\n\nHeadline: Scientists develop biodegradable plastic that dissolves in seawater.\nClass: SOLUTION\n\nHeadline: Record number of species added to endangered list in 2024.\nClass:PROBLEM\n\nHeadline: New study reveals how climate change is affecting the spread of diseases.\nClass: SOLUTION\n\nHeadline: New study shows how climate change is affecting the spread of diseases.\nClass: SOLUTION\n\nHeadline: New study shows how climate change is affecting the spread of diseases.\nClass: SOLUTION\n\nHeadline: New study shows how climate change is affecting the spread of diseases.\nClass: SOLUTION\n\nHeadline: New study shows how climate change is affecting the 

### 1.3 Chain-of-Thought Prompting
Asking the model to "think step by step" improves accuracy on tasks that require reasoning.

In [7]:
article2 = """
A government report released today shows that unemployment has dropped to 3.8%,
the lowest in 20 years. However, inflation remains at 6.2%, meaning many workers
are seeing their real wages fall despite being employed. Consumer confidence
surveys show mixed results, with younger households reporting greater financial stress.
"""

# Without chain-of-thought
ask(f"Is the economic situation described in this article good or bad?\n{article2}")

[transformers] Both `max_new_tokens` (=200) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


PROMPT:
Is the economic situation described in this article good or bad?

A government report released today shows that unemployment has dropped to 3.8%,
the lowest in 20 years. However, inflation remains at 6.2%, meaning many workers
are seeing their real wages fall despite being employed. Consumer confidence
surveys show mixed results, with younger households reporting greater financial stress.


RESPONSE:
The article also mentions that the government is considering a new tax on the wealthy,
which could be a significant blow to the middle class. The report also notes that
the government is struggling to balance its budget, with a projected deficit of
$1.5 billion for the current fiscal year.

Overall, the economic situation in the United States is complex and uncertain,
with many factors contributing to the current state of affairs.
------------------------------------------------------------


'The article also mentions that the government is considering a new tax on the wealthy,\nwhich could be a significant blow to the middle class. The report also notes that\nthe government is struggling to balance its budget, with a projected deficit of\n$1.5 billion for the current fiscal year.\n\nOverall, the economic situation in the United States is complex and uncertain,\nwith many factors contributing to the current state of affairs.'

In [8]:
# With chain-of-thought
ask(f"Is the economic situation described in this article good or bad? "
    f"Think through the evidence step by step before giving your verdict.\n{article2}")

[transformers] Both `max_new_tokens` (=200) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


PROMPT:
Is the economic situation described in this article good or bad? Think through the evidence step by step before giving your verdict.

A government report released today shows that unemployment has dropped to 3.8%,
the lowest in 20 years. However, inflation remains at 6.2%, meaning many workers
are seeing their real wages fall despite being employed. Consumer confidence
surveys show mixed results, with younger households reporting greater financial stress.


RESPONSE:
The report also shows that the economy is growing at a slower pace than in previous years.

The government's economic situation report is a critical indicator of the state of the economy. It provides valuable insights into the state of the economy and helps policymakers make informed decisions.

However, the report's findings are not without their flaws. The report's methodology is not always consistent, and it may not be representative of the entire population. Additionally, the report's data may not be fully accu

"The report also shows that the economy is growing at a slower pace than in previous years.\n\nThe government's economic situation report is a critical indicator of the state of the economy. It provides valuable insights into the state of the economy and helps policymakers make informed decisions.\n\nHowever, the report's findings are not without their flaws. The report's methodology is not always consistent, and it may not be representative of the entire population. Additionally, the report's data may not be fully accurate, as it relies on surveys and other sources of information.\n\nOverall, the economic situation described in this article is not good, as it shows a slowing economy and rising inflation. While the report's findings are important, they should be viewed with caution and should be used in conjunction with other data and analysis."

### 1.4 Structured Output
Ask the model to return data in a structured format so you can use it in downstream code.

In [9]:
import json

article3 = """
The European Central Bank raised interest rates by 0.25% on Thursday, bringing
the benchmark rate to 4.5%. ECB President Christine Lagarde stated the decision
was driven by persistent inflation across the eurozone. Markets reacted negatively,
with the Euro Stoxx 50 index falling 1.8% on the day.
"""

structured_prompt = f"""
Extract information from this news article and return ONLY a JSON object with these keys:
- who: the main person or organisation
- what: what they did
- impact: the stated impact or reaction

Article: {article3}
JSON:"""

response = ask(structured_prompt)
try:
    print("\nParsed JSON:")
    print(json.dumps(json.loads(response), indent=2))
except:
    print("\n(Tip: frontier models like GPT-4 or Claude return reliable JSON — small models sometimes don't.)")

[transformers] Both `max_new_tokens` (=200) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


PROMPT:

Extract information from this news article and return ONLY a JSON object with these keys:
- who: the main person or organisation
- what: what they did
- impact: the stated impact or reaction

Article: 
The European Central Bank raised interest rates by 0.25% on Thursday, bringing
the benchmark rate to 4.5%. ECB President Christine Lagarde stated the decision
was driven by persistent inflation across the eurozone. Markets reacted negatively,
with the Euro Stoxx 50 index falling 1.8% on the day.

JSON:

RESPONSE:
Extract information from this news article and return ONLY a JSON object with these keys:
- who: the main person or organisation
- what: what they did
- impact: the stated impact or reaction

Article: 
The European Central Bank raised interest rates by 0.25% on Thursday, bringing
the benchmark rate to 4.5%. ECB President Christine Lagarde stated the decision
was driven by persistent inflation across the eurozone. Markets reacted negatively,
with the Euro Stoxx 50 index 

### Tips for writing good prompts

**Do:**
- Be specific about format: *"Answer in one sentence"*, *"Return a JSON object"*
- Specify the audience: *"Explain to someone with no economics background"*
- Give a role: *"You are a journalist summarising for a general audience..."*
- Iterate — if the first attempt fails, refine the prompt

**Avoid:**
- Vague instructions: *"Tell me about this article"*
- Trusting the model on facts without verification
- Assuming the model knows your specific context

### Exercise
Copy a paragraph from a news article you read recently. Write a prompt to extract: the main topic, the key people mentioned, and the most important number or statistic.

In [10]:
# YOUR TURN
my_article = """
UK advertising giant WPP has posted larger-than-expected annual profits and predicted that it will outperform the market in 2005. Pre-tax profits rose 15% from a year ago to reach £546m ($1.04bn), ahead of average analysts' forecasts of £532m. Chief Executive Martin Sorrell told Reuters that WPP had submitted a proposal for United Business Media's NOP World market research unit.
"""

my_prompt = f"""
Read the article below and extract the following information.

Return your answer in valid JSON with exactly these keys:
- "main_topic"
- "key_people"
- "important_number_or_statistic"

Rules:
- main_topic: one short sentence
- key_people: list of person names mentioned
- important_number_or_statistic: the single most important number or statistic, with brief context

Article:
{my_article}
"""

ask(my_prompt)

[transformers] Both `max_new_tokens` (=200) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


PROMPT:

Read the article below and extract the following information.

Return your answer in valid JSON with exactly these keys:
- "main_topic"
- "key_people"
- "important_number_or_statistic"

Rules:
- main_topic: one short sentence
- key_people: list of person names mentioned
- important_number_or_statistic: the single most important number or statistic, with brief context

Article:

UK advertising giant WPP has posted larger-than-expected annual profits and predicted that it will outperform the market in 2005. Pre-tax profits rose 15% from a year ago to reach £546m ($1.04bn), ahead of average analysts' forecasts of £532m. Chief Executive Martin Sorrell told Reuters that WPP had submitted a proposal for United Business Media's NOP World market research unit.



RESPONSE:
Read the article below and extract the following information.

Return your answer in valid JSON with exactly these keys:
- "main_topic"
- "key_people"
- "important_number_or_statistic"

Rules:
- main_topic: one shor

'Read the article below and extract the following information.\n\nReturn your answer in valid JSON with exactly these keys:\n- "main_topic"\n- "key_people"\n- "important_number_or_statistic"\n\nRules:\n- main_topic: one short sentence\n- key_people: list of person names mentioned\n- important_number_or_statistic: the single most important number or statistic, with brief context\n\nArticle:\n\nUK advertising giant WPP has posted larger-than-expected annual profits and predicted that it will outperform the market in 2005. Pre-tax profits rose 15% from a year ago to reach £546m ($1.04bn), ahead of average analysts\' forecasts of £532m. Chief Executive Martin Sorrell told Reuters that WPP had submitted a proposal for United Business Media\'s NOP World market research unit.\n\nThe company said it would pay £100m for the unit, which includes the NOP World and NOP Europe brands, and would pay £100m for the NOP Asia Pacific business.\n\nWPP said it would pay £100m for the NOP World and NOP Eur

---
# MODULE 2: Fine-Tuning

## The idea
A pre-trained model like BERT has read billions of words from the internet. Fine-tuning takes that broad knowledge and **specialises it for your specific task** using a small labelled dataset you provide.

**Analogy:** A new doctor has broad medical knowledge (pre-training). A residency in cardiology specialises them for a specific domain (fine-tuning). They don't re-learn medicine from scratch.

### When to fine-tune vs just prompt?
| Situation | Use |
|---|---|
| General task, standard labels | Prompt engineering |
| Custom label scheme specific to your work | Fine-tune |
| Need consistent, repeatable outputs at scale | Fine-tune |
| Have labelled examples (even 200–500) | Fine-tune |

### What we'll do
Fine-tune a model to classify news articles into four categories: **Politics, Technology, Sports, Business.**

In [11]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from datasets import Dataset
import numpy as np

# --- Step 1: Labelled dataset ---
# In a real project you'd have hundreds of these — we use 20 to demonstrate the pipeline

data = {
    "text": [
        # Politics (label 0)
        "The prime minister announced a new cabinet reshuffle following last week's election results.",
        "Parliament voted to approve the emergency budget after weeks of cross-party negotiations.",
        "Diplomatic tensions rose after the ambassador was recalled over the trade dispute.",
        "The opposition party filed a motion of no confidence ahead of the summer recess.",
        "Voters in three key swing states will decide the outcome of Tuesday's referendum.",
        # Technology (label 1)
        "The company unveiled a new AI chip claimed to be ten times faster than its predecessor.",
        "Regulators opened an investigation into the social media platform over data privacy concerns.",
        "A major software update patched a critical security vulnerability affecting millions of users.",
        "The electric vehicle startup raised $2 billion in its latest funding round.",
        "Scientists demonstrated a quantum computer solving a problem in minutes that would take classical computers years.",
        # Sports (label 2)
        "The national team secured a place in the World Cup finals after a dramatic penalty shootout.",
        "The record transfer fee was broken again as the striker moved to the Spanish club for $200 million.",
        "Bad weather forced the postponement of three matches scheduled for the weekend.",
        "The veteran athlete announced her retirement after winning her fifth Olympic gold medal.",
        "Controversial refereeing decisions overshadowed an otherwise thrilling championship final.",
        # Business (label 3)
        "Quarterly earnings beat analyst expectations as the retailer reported 12% revenue growth.",
        "The central bank held interest rates steady amid signs of slowing economic growth.",
        "Thousands of jobs will be cut as the airline restructures its operations following losses.",
        "The merger between the two pharmaceutical giants is expected to close by end of year.",
        "Oil prices fell sharply after OPEC announced a surprise increase in production quotas.",
    ],
    "label": [0,0,0,0,0, 1,1,1,1,1, 2,2,2,2,2, 3,3,3,3,3]
}

label_names = ["Politics", "Technology", "Sports", "Business"]

dataset = Dataset.from_dict(data).train_test_split(test_size=0.2, seed=42)
print(f"Train: {len(dataset['train'])} examples | Test: {len(dataset['test'])} examples")
print(f"\nExample:")
print(f"  Text:  {dataset['train'][0]['text']}")
print(f"  Label: {label_names[dataset['train'][0]['label']]}")

Train: 16 examples | Test: 4 examples

Example:
  Text:  Bad weather forced the postponement of three matches scheduled for the weekend.
  Label: Sports


In [12]:
# --- Step 2: Tokenise ---
# Converts words into numbers the model understands

MODEL_NAME = "distilbert-base-uncased"  # Smaller, faster version of BERT
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

tokenized = dataset.map(
    lambda batch: tokenizer(batch["text"], padding=True, truncation=True, max_length=128),
    batched=True
)

# What does tokenisation look like?
example = tokenizer("Oil prices fell after OPEC increased production.")
print("Original: 'Oil prices fell after OPEC increased production.'")
print(f"Token IDs: {example['input_ids']}")
print(f"Decoded:   {tokenizer.decode(example['input_ids'])}")

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/16 [00:00<?, ? examples/s]

Map:   0%|          | 0/4 [00:00<?, ? examples/s]

Original: 'Oil prices fell after OPEC increased production.'
Token IDs: [101, 3514, 7597, 3062, 2044, 6728, 8586, 3445, 2537, 1012, 102]
Decoded:   [CLS] oil prices fell after opec increased production. [SEP]


In [13]:
# --- Step 3: Load model ---
# DistilBERT + a classification head on top

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=4,
    id2label={i: name for i, name in enumerate(label_names)},
    label2id={name: i for i, name in enumerate(label_names)}
)

total = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {total:,}")
print("We're fine-tuning a 67M parameter model using just 16 training examples.")
print("This works because the model already understands language — we're just redirecting it.")

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model parameters: 66,956,548
We're fine-tuning a 67M parameter model using just 16 training examples.
This works because the model already understands language — we're just redirecting it.


In [14]:
# --- Step 4: Train ---

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {"accuracy": (preds == labels).mean()}

training_args = TrainingArguments(
    output_dir="./news_classifier",
    num_train_epochs=6,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    eval_strategy="epoch",
    save_strategy="no",
    logging_steps=5,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["test"],
    compute_metrics=compute_metrics,
)

print("Training...")
trainer.train()
print("\nDone!")

Training...


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy
1,No log,1.354039,0.250000
2,No log,1.317675,0.750000
3,1.269505,1.279631,0.750000
4,1.269505,1.251164,0.750000
5,0.987149,1.227387,0.750000
6,0.987149,1.216468,0.750000



Done!


In [15]:
# --- Step 5: Test on new headlines ---
from transformers import pipeline as hf_pipeline

classifier = hf_pipeline("text-classification", model=model, tokenizer=tokenizer)

new_headlines = [
    "The senate passed a new climate bill with bipartisan support.",
    "The smartphone maker reported record profits despite supply chain issues.",
    "The marathon runner broke the world record by four seconds.",
    "Inflation fell to its lowest level in two years according to official figures.",
]

print("Predictions on new headlines (not seen during training):")
print("-" * 65)
for h in new_headlines:
    result = classifier(h)[0]
    print(f"Headline:  {h}")
    print(f"Predicted: {result['label']} (confidence: {result['score']:.1%})\n")

Predictions on new headlines (not seen during training):
-----------------------------------------------------------------
Headline:  The senate passed a new climate bill with bipartisan support.
Predicted: Politics (confidence: 46.0%)

Headline:  The smartphone maker reported record profits despite supply chain issues.
Predicted: Technology (confidence: 32.6%)

Headline:  The marathon runner broke the world record by four seconds.
Predicted: Sports (confidence: 29.8%)

Headline:  Inflation fell to its lowest level in two years according to official figures.
Predicted: Business (confidence: 35.7%)



### Key takeaways
- We trained on **16 examples** and it works — in practice, 500+ per class is better
- Training took minutes, not hours, because we started from a pre-trained model
- The same pipeline works for any classification task you care about in your research

### Exercise
Add a 5th category to the `data` dictionary — for example, **Health** or **Environment**. Add 5 example sentences for it and retrain. Does the model pick it up?

---
# MODULE 3: Retrieval-Augmented Generation (RAG)

## The idea
LLMs don't know your specific documents. RAG solves this by letting you attach your own knowledge base — papers, reports, articles — to the model at query time.

```
Your Question
     ↓
[ RETRIEVE ]  Search your document library for relevant passages
     ↓
[ AUGMENT  ]  Add those passages to the prompt as context
     ↓
[ GENERATE ]  LLM answers using the retrieved context
```

**Analogy:** An open-book exam. The model doesn't need to memorise everything — it looks things up at question time.

**Why it matters:** Less hallucination, answers grounded in your actual documents, no retraining needed.

In [16]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

# --- Step 1: A library of news article passages ---
# Imagine these are paragraphs extracted from 100 articles you want to query

documents = [
    # Climate
    "Global average temperatures in 2024 exceeded pre-industrial levels by 1.5C for the first time on record. "
    "The UN Secretary-General described the milestone as a moment of reckoning for world leaders.",

    "The Amazon rainforest lost a record 11,000 square kilometres of tree cover in the first half of 2024, "
    "driven by agricultural expansion and illegal logging, according to satellite data.",

    "A new carbon capture facility opened in Iceland, capable of removing 36,000 tonnes of CO2 per year "
    "directly from the air. Critics argue the cost of $1,000 per tonne remains far too high for scale.",

    # Economy
    "The International Monetary Fund revised global growth forecasts downward to 2.9% for 2025, "
    "citing persistent inflation, high interest rates, and slowing trade as the main headwinds.",

    "Unemployment in the eurozone fell to 6.0% in October, the lowest since the euro was introduced. "
    "However, youth unemployment remains disproportionately high at 14.3% across member states.",

    "Central banks in the US and Europe began cutting interest rates in late 2024 after two years of "
    "aggressive hikes. Analysts expect borrowing costs to fall by a further 1% through 2025.",

    # Technology
    "A major AI laboratory released a model capable of solving university-level mathematics problems "
    "with 90% accuracy, a significant jump from the 50% recorded just two years earlier.",

    "Cyberattacks on critical infrastructure rose by 38% in 2024, with energy grids and water treatment "
    "facilities among the most frequently targeted, according to a new cybersecurity report.",

    "The global semiconductor shortage has largely eased, with chip production capacity expanding "
    "significantly after $200 billion in government subsidies were directed at domestic manufacturing.",

    # Health
    "A clinical trial involving 10,000 participants found that a new mRNA vaccine reduced the risk "
    "of a common respiratory illness by 74%, with no serious side effects reported.",

    "The World Health Organization declared antimicrobial resistance a top global health threat, "
    "warning that drug-resistant infections could cause 10 million deaths annually by 2050.",

    "Rates of depression and anxiety among adults aged 18-34 have doubled since 2015, according to "
    "a large longitudinal study, with social media use cited as a contributing factor.",

    # Politics
    "Turnout in last month's general election reached 71%, the highest in three decades, driven largely "
    "by record participation among first-time voters aged 18-24.",

    "The G7 summit concluded with a joint statement pledging to phase out coal power by 2035, "
    "though critics noted the agreement contained no binding enforcement mechanism.",
]

print(f"Document library: {len(documents)} passages loaded")

Document library: 14 passages loaded


In [17]:
# --- Step 2: Convert documents to vectors (embeddings) ---
# Each passage becomes a list of numbers that captures its meaning
# Similar passages get similar vectors — this is how semantic search works

embed_model = SentenceTransformer('all-MiniLM-L6-v2')

print("Embedding documents...")
doc_embeddings = embed_model.encode(documents, show_progress_bar=True)

print(f"\nEach passage is now a vector of {doc_embeddings.shape[1]} numbers.")
print("Passages with similar meaning will have vectors close together in space.")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding documents...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]


Each passage is now a vector of 384 numbers.
Passages with similar meaning will have vectors close together in space.


In [18]:
# --- Step 3: Build a searchable index ---
# FAISS lets us find the most similar vectors to a query in milliseconds

index = faiss.IndexFlatL2(doc_embeddings.shape[1])
index.add(doc_embeddings.astype('float32'))

def retrieve(query, top_k=3):
    vec = embed_model.encode([query]).astype('float32')
    distances, indices = index.search(vec, top_k)
    return [documents[i] for i in indices[0]]

# Test retrieval alone first — does it find the right passages?
print("Query: 'What is happening with global temperatures?'\n")
for i, doc in enumerate(retrieve("What is happening with global temperatures?")):
    print(f"[{i+1}] {doc[:110]}...\n")

Query: 'What is happening with global temperatures?'

[1] Global average temperatures in 2024 exceeded pre-industrial levels by 1.5C for the first time on record. The U...

[2] The International Monetary Fund revised global growth forecasts downward to 2.9% for 2025, citing persistent i...

[3] The global semiconductor shortage has largely eased, with chip production capacity expanding significantly aft...



In [19]:
# --- Step 4: Build the full RAG pipeline ---

from transformers import pipeline as hf_pipeline
from sentence_transformers import util

qa_model = hf_pipeline("text-generation", model="TinyLlama/TinyLlama-1.1B-Chat-v1.0", max_new_tokens=150, do_sample=False)

# Minimum cosine similarity for a passage to count as relevant
RELEVANCE_THRESHOLD = 0.25

def rag_answer(question, verbose=True):
    # Step 1: Retrieve top passages
    passages = retrieve(question, top_k=3)

    # Step 2: Score relevance — don't trust the LLM to self-police
    # If nothing in the library is semantically close to the question, say so
    q_vec = embed_model.encode(question, convert_to_tensor=True)
    p_vecs = embed_model.encode(passages, convert_to_tensor=True)
    scores = util.cos_sim(q_vec, p_vecs)[0]
    best_score = scores.max().item()

    if best_score < RELEVANCE_THRESHOLD:
        if verbose:
            print(f"QUESTION: {question}")
            print(f"Best relevance score: {best_score:.2f} (below threshold {RELEVANCE_THRESHOLD})")
            print("ANSWER: I don't have that information in my document library.")
            print("-" * 65)
        return "I don't have that information in my document library."

    # Step 3: Build prompt
    context = "\n".join([f"- {p}" for p in passages])
    prompt = (
        "Answer the question using only the context below.\n"
        "If the answer is not present, say: I do not have that information.\n\n"
        f"Context:\n{context}\n\n"
        f"Question: {question}\nAnswer:"
    )

    # Step 4: Generate
    full = qa_model(prompt)[0]["generated_text"]
    answer = full.split("Answer:")[-1].strip()

    if verbose:
        print(f"QUESTION: {question}")
        print(f"\nRETRIEVED PASSAGES (best relevance: {best_score:.2f}):")
        for i, (p, s) in enumerate(zip(passages, scores)):
            print(f"  [{i+1}] score={s:.2f}  {p[:85]}...")
        print(f"\nANSWER: {answer}")
        print("-" * 65)
    return answer

print("RAG pipeline ready!")


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

RAG pipeline ready!


In [20]:
# --- Step 5: Ask questions across the document library ---

questions = [
    "What is the current state of global economic growth?",
    "What are the main threats to public health right now?",
    "What progress has been made on climate change?",
]

for q in questions:
    rag_answer(q)
    print()

[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


QUESTION: What is the current state of global economic growth?

RETRIEVED PASSAGES (best relevance: 0.55):
  [1] score=0.55  The International Monetary Fund revised global growth forecasts downward to 2.9% for ...
  [2] score=0.35  The global semiconductor shortage has largely eased, with chip production capacity ex...
  [3] score=0.31  Global average temperatures in 2024 exceeded pre-industrial levels by 1.5C for the fi...

ANSWER: The global economy is expected to grow at a slower pace in 2025, with the International Monetary Fund revised downward its forecast to 2.9%. The global semiconductor shortage has largely eased, with chip production capacity expanding significantly after $200 billion in government subsidies were directed at domestic manufacturing. Global average temperatures in 2024 exceeded pre-industrial levels by 1.5C for the first time on record, with the UN Secretary-General describing the milestone as a moment of reckoning for world leaders.
---------------------------

[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


QUESTION: What are the main threats to public health right now?

RETRIEVED PASSAGES (best relevance: 0.44):
  [1] score=0.44  The World Health Organization declared antimicrobial resistance a top global health t...
  [2] score=0.26  Cyberattacks on critical infrastructure rose by 38% in 2024, with energy grids and wa...
  [3] score=0.23  A clinical trial involving 10,000 participants found that a new mRNA vaccine reduced ...

ANSWER: Antimicrobial resistance, cyberattacks on critical infrastructure, and a clinical trial that found a new mRNA vaccine reduced the risk of a common respiratory illness by 74%.
-----------------------------------------------------------------

QUESTION: What progress has been made on climate change?

RETRIEVED PASSAGES (best relevance: 0.43):
  [1] score=0.43  Global average temperatures in 2024 exceeded pre-industrial levels by 1.5C for the fi...
  [2] score=0.30  The International Monetary Fund revised global growth forecasts downward to 2.9% for ...
  [3]

In [21]:
# --- Step 6: Why RAG matters — with vs without context ---

question = "What is the state of youth unemployment in Europe?"

# Without RAG
no_rag = qa_model(f"Answer this question: {question}")[0]['generated_text'].split('Answer this question:')[-1].strip()

# With RAG
with_rag = rag_answer(question, verbose=False)

print("WITHOUT RAG (model guesses from training data):")
print(f"  {no_rag}")
print()
print("WITH RAG (answer grounded in our documents):")
print(f"  {with_rag}")
print()
print("RAG gives specific, verifiable answers instead of generic ones.")

[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


WITHOUT RAG (model guesses from training data):
  What is the state of youth unemployment in Europe?

WITH RAG (answer grounded in our documents):
  Youth unemployment in Europe has remained high, with 14.3% of 18-34-year-olds unemployed in 2019, according to a large longitudinal study.

RAG gives specific, verifiable answers instead of generic ones.


In [22]:
# --- Step 7: What happens when the answer isn't in the library? ---
# A well-behaved RAG system should say so, not hallucinate

rag_answer("Who won the most recent Formula 1 World Championship?")

QUESTION: Who won the most recent Formula 1 World Championship?
Best relevance score: 0.21 (below threshold 0.25)
ANSWER: I don't have that information in my document library.
-----------------------------------------------------------------


"I don't have that information in my document library."

### Key takeaways
- RAG = a search engine connected to a language model
- Your documents stay on your machine — important for sensitive or proprietary data
- Works best when document chunks are 100–300 words — not too long, not too short
- In practice, tools like **LangChain** and **LlamaIndex** handle the plumbing for you

### Exercise
Add 3–5 paragraphs from a news story you know well to the `documents` list. Rebuild the index and ask a question you already know the answer to. Does RAG retrieve the right passage?

In [23]:
# YOUR TURN
my_docs = [
    "Paste a news paragraph here.",
    "And another one here.",
]

all_docs = documents + my_docs
all_embeddings = embed_model.encode(all_docs).astype('float32')
new_index = faiss.IndexFlatL2(all_embeddings.shape[1])
new_index.add(all_embeddings)

my_question = "Write your question here"
vec = embed_model.encode([my_question]).astype('float32')
_, idxs = new_index.search(vec, 3)

print(f"Top results for: '{my_question}'\n")
for i, idx in enumerate(idxs[0]):
    print(f"[{i+1}] {all_docs[idx][:120]}...\n")

Top results for: 'Write your question here'

[1] Paste a news paragraph here....

[2] And another one here....

[3] The global semiconductor shortage has largely eased, with chip production capacity expanding significantly after $200 bi...



---
# Summary: Which tool to use when?

| Your situation | Best approach |
|---|---|
| One-off task: summarise, translate, reformat | **Prompt Engineering** |
| Classify text using your own label scheme | **Fine-Tuning** |
| Answer questions across a collection of documents | **RAG** |
| Don't have labelled data yet | **Prompt Engineering** to generate labels, then **Fine-Tune** |
| Documents change frequently | **RAG** (no retraining needed) |


---